<a href="https://colab.research.google.com/github/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/compare-jev-bigquery-ai-functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BigQueryからJevとGeminiを比較 / Jev vs Gemini from BigQuery

## 要約 / tl;dr

- Jev：正解率85.5%、200件で$0.00788。 / Jev: 85.5% accuracy at $0.00788 for 200 rows.
- Gemini 2.5 Flash-Lite：86.0%、$0.01554。 / Gemini 2.5 Flash-Lite: 86.0% at $0.01554.
- p50はJevが545ms、Flash-Liteが1,950msで、Jevが約3.6倍高速。 / Jev's p50 was 545ms vs 1,950ms, about 3.6× faster.
- Gemini 3.1 Pro Preview：88.5%、$0.55055。 / Gemini 3.1 Pro Preview: 88.5% at $0.55055.
- 広いPro再判定は費用対効果を改善しませんでした。 / Broad Pro reranking did not improve cost-effectiveness.

主比較はすべて同じVercel AI Gateway経路です。このNotebookには集計結果だけを収録し、質問本文や認証情報は含みません。

All primary comparisons use the same Vercel AI Gateway path. This notebook contains aggregate results only, not question text or credentials.

## 背景と方法 / Context & Methods

### 主な前提 / Key assumptions

- 元テーブル / Source: [`bigquery-public-data.stackoverflow.posts_questions`](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=stackoverflow&t=posts_questions&page=table)
- [Google CloudのKeras例](https://cloud.google.com/blog/products/gcp/intro-to-text-classification-with-keras-automatically-tagging-stack-overflow-posts)と同じ上位20タグの単一ラベル分類です。 / This is the same top-20, single-label task as Google Cloud's Keras example.
- 20タグ × 各10問を決定的に抽出した200件です。 / The set contains 20 tags × 10 deterministically selected questions.
- 全モデルの経路：BigQuery → Remote Function → Cloud Run → Vercel AI Gateway → model。 / Every primary model uses this same route.
- 質問文、指示、候補名、候補説明は同一です。 / Question text, instructions, candidate names, and descriptions are identical.
- 費用は実測token数 × 公開単価です。インフラ、失敗、再試行、クレジットは除外します。 / Cost is observed tokens × public price; infrastructure, failures, retries, and credits are excluded.

## データ / Data

質問本文は再配布しません。[`sql/01_prepare_stackoverflow.sql`](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/sql/01_prepare_stackoverflow.sql)で公開BigQueryテーブルから評価データを再構築できます。

Question text is not redistributed. [`sql/01_prepare_stackoverflow.sql`](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/sql/01_prepare_stackoverflow.sql) rebuilds the evaluation set from the public BigQuery table.

## 参照ファイル / Repository files

- [READMEと全体手順 / README and overview](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/README.md)
- [評価データ作成SQL / Build the evaluation sample](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/sql/01_prepare_stackoverflow.sql)
- [BigQuery接続・Remote Function作成SQL / Create connections and Remote Functions](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/sql/02_create_remote_functions.sql)
- [Jev実行SQL / Run Jev](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/sql/03_run_jev.sql)
- [Gemini 2.5 Flash-Lite実行SQL / Run Gemini 2.5 Flash-Lite](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/sql/04_run_gateway_gemini25fl.sql)
- [Pro・二段階判定SQL / Pro and two-stage experiments](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/sql/05_run_pro_experiments.sql)
- [Cloud Runアダプター / Cloud Run adapter](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/tree/main/cloud-run)
- [環境変数例 / Environment variable template](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/.env.example)
- [集計結果CSV / Aggregate results](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/results/summary.csv)
- [速度・ジョブ時間CSV / Latency and job-time results](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/results/performance.csv)
- [ブログ用ヒートマップPNG / Blog heatmap PNG](https://github.com/jackojacko05/compare-jev-bigquery-ai-functions/blob/main/docs/assets/blog-tradeoff-heatmap.png)

認証情報は含みません。Vercel AI GatewayのキーはGoogle Secret Managerなどで管理してください。 / No credentials are included. Store the Vercel AI Gateway key in Google Secret Manager or an equivalent secret store.

## Colabだけでライブ再実行 / End-to-end live reproduction

以下は任意のライブ再実行です。集計済み結果を見るだけなら実行不要です。Jevも、Cloud RunアダプターとBigQuery Remote FunctionをNotebookから構築すれば同じ経路で実行できます。

Live reproduction is optional. It provisions the Jev/Gemini adapter on Cloud Run, creates BigQuery Remote Functions, prepares the public-data sample, and runs inference.

### 事前準備 / Prerequisites

1. 課金が有効なGoogle Cloudプロジェクトを用意します。 / Use a billing-enabled Google Cloud project.
2. Colab左側の鍵アイコン「シークレット」で `VERCEL_AI_GATEWAY_API_KEY` を追加し、このNotebookからのアクセスを許可します。値はNotebookやGitHubへ書きません。 / In Colab Secrets, add `VERCEL_AI_GATEWAY_API_KEY` and allow notebook access. Never paste it into a cell or GitHub.
3. 最初は `RUN_LIVE = False` のまま設定確認し、費用と権限を確認してから `True` にします。 / Keep `RUN_LIVE = False` until you have reviewed permissions and cost.

必要権限の目安：BigQuery Admin、Cloud Run Admin、Service Account User、Service Usage Admin、Secret Manager Admin。組織環境では最小権限へ分割してください。

Typical setup roles are BigQuery Admin, Cloud Run Admin, Service Account User, Service Usage Admin, and Secret Manager Admin. Split these into least-privilege roles in managed environments.

> **費用注意 / Cost warning:** Remote Functionは再試行されることがあります。200件を物理テーブルへ固定し、run IDごとに結果を保存してから実行してください。 / Remote Functions may retry. Materialize the 200 rows and persist results by run ID.

In [1]:
# 設定 / Configuration
PROJECT_ID = "YOUR_PROJECT_ID"       # 例 / example: my-gcp-project
REGION = "us-central1"               # Cloud Run region
BQ_LOCATION = "US"                   # BigQuery dataset/connection location
DATASET = "jev_benchmark"
SERVICE = "jev-bigquery-adapter"
CONNECTION = "jev_remote_connection"
SECRET = "vercel-ai-gateway-key"
RUN_LIVE = False  # 課金とリソース作成を理解した後だけ True / Set True only after review

assert not RUN_LIVE or PROJECT_ID != "YOUR_PROJECT_ID", "Set PROJECT_ID before live execution"
print({"project": PROJECT_ID, "region": REGION, "bq_location": BQ_LOCATION, "run_live": RUN_LIVE})

{'project': 'YOUR_PROJECT_ID', 'region': 'us-central1', 'bq_location': 'US', 'run_live': False}


In [2]:
# 認証、Secret Manager、BigQuery Connection、Cloud Run / Auth and infrastructure
if RUN_LIVE:
    import json, subprocess
    from pathlib import Path
    from google.colab import auth, userdata

    auth.authenticate_user()
    gateway_key = userdata.get("VERCEL_AI_GATEWAY_API_KEY")
    if not gateway_key:
        raise ValueError("Add VERCEL_AI_GATEWAY_API_KEY in Colab Secrets and enable notebook access")

    def run(*args, input_text=None):
        return subprocess.run(args, input=input_text, text=True, check=True, capture_output=True).stdout.strip()

    run("gcloud", "config", "set", "project", PROJECT_ID)
    run("gcloud", "services", "enable", "bigquery.googleapis.com", "bigqueryconnection.googleapis.com",
        "run.googleapis.com", "cloudbuild.googleapis.com", "artifactregistry.googleapis.com",
        "secretmanager.googleapis.com")
    subprocess.run(["bq", "--location", BQ_LOCATION, "mk", "--dataset", f"{PROJECT_ID}:{DATASET}"], check=False)
    subprocess.run(["bq", "mk", "--connection", "--location", BQ_LOCATION,
                    "--connection_type=CLOUD_RESOURCE", CONNECTION], check=False)

    # Create or update the secret without printing it.
    exists = subprocess.run(["gcloud", "secrets", "describe", SECRET], capture_output=True).returncode == 0
    if not exists:
        run("gcloud", "secrets", "create", SECRET, "--replication-policy=automatic")
    run("gcloud", "secrets", "versions", "add", SECRET, "--data-file=-", input_text=gateway_key)
    del gateway_key

    # Use an explicit runtime identity and allow it to read only this secret.
    project_number = run("gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)")
    runtime_sa = f"{project_number}-compute@developer.gserviceaccount.com"
    run("gcloud", "secrets", "add-iam-policy-binding", SECRET,
        "--member", f"serviceAccount:{runtime_sa}", "--role", "roles/secretmanager.secretAccessor")

    repo_dir = Path("/content/compare-jev-bigquery-ai-functions")
    if not repo_dir.exists():
        run("git", "clone", "--depth=1", "https://github.com/jackojacko05/compare-jev-bigquery-ai-functions.git", str(repo_dir))
    else:
        run("git", "-C", str(repo_dir), "pull", "--ff-only")

    run("gcloud", "run", "deploy", SERVICE, "--source", str(repo_dir / "cloud-run"),
        "--region", REGION, "--no-allow-unauthenticated", "--service-account", runtime_sa,
        "--set-secrets", f"AI_GATEWAY_API_KEY={SECRET}:latest", "--quiet")
    service_url = run("gcloud", "run", "services", "describe", SERVICE, "--region", REGION,
                      "--format=value(status.url)")
    connection_json = run("bq", "show", "--connection", "--format=prettyjson",
                          f"{PROJECT_ID}.{BQ_LOCATION.lower()}.{CONNECTION}")
    connection_sa = json.loads(connection_json)["cloudResource"]["serviceAccountId"]
    run("gcloud", "run", "services", "add-iam-policy-binding", SERVICE, "--region", REGION,
        "--member", f"serviceAccount:{connection_sa}", "--role", "roles/run.invoker")
    print({"service_url": service_url, "connection_service_account": connection_sa})
else:
    print("Skipped infrastructure creation / インフラ作成をスキップ")

Skipped infrastructure creation / インフラ作成をスキップ


In [3]:
# 評価データとRemote Functionを作成 / Prepare data and Remote Functions
if RUN_LIVE:
    from google.cloud import bigquery

    client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
    repo_dir = Path("/content/compare-jev-bigquery-ai-functions")

    def sql_file(name, cloud_run_url=None):
        sql = (repo_dir / "sql" / name).read_text()
        sql = sql.replace("YOUR_PROJECT_ID", PROJECT_ID)
        if cloud_run_url:
            sql = sql.replace("https://YOUR_CLOUD_RUN_URL", cloud_run_url)
        return sql

    client.query(sql_file("01_prepare_stackoverflow.sql")).result()
    client.query(sql_file("02_create_remote_functions.sql", service_url)).result()
    counts = client.query(
        f"SELECT COUNT(*) rows, COUNT(DISTINCT label) labels "
        f"FROM `{PROJECT_ID}.{DATASET}.so20_pilot200`"
    ).to_dataframe()
    display(counts)
else:
    print("Skipped data preparation / データ作成をスキップ")

Skipped data preparation / データ作成をスキップ


In [4]:
# 200件のJevとGeminiを実行 / Run Jev and Gemini on 200 rows
# このセルから外部推論費用が発生します。 / External inference cost starts here.
if RUN_LIVE:
    RUN_JEV = True
    RUN_GEMINI = True
    jobs = [
        (RUN_JEV, "so20_jev_gateway_200_v1", "03_run_jev.sql"),
        (RUN_GEMINI, "so20_gateway_gemini25fl_full_official_200_20260920_01", "04_run_gateway_gemini25fl.sql"),
    ]
    for enabled, run_id, filename in jobs:
        if not enabled:
            continue
        existing = next(iter(client.query(
            f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.{DATASET}.so20_results` WHERE run_id = @run_id",
            job_config=bigquery.QueryJobConfig(query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", run_id)
            ]),
        ).result())).n
        if existing:
            print(f"Skip {run_id}: {existing} persisted rows already exist")
        else:
            client.query(sql_file(filename)).result()
            print(f"Completed {run_id}")
else:
    print("Skipped paid inference / 有料推論をスキップ")

Skipped paid inference / 有料推論をスキップ


In [5]:
# ライブ結果を集計 / Summarize live results
if RUN_LIVE:
    live_summary = client.query(f'''
      SELECT run_id, model,
             COUNT(*) AS rows,
             COUNTIF(error IS NULL) AS completed,
             AVG(CAST(predicted_label = actual_label AS INT64)) AS accuracy,
             SUM(provider_cost_usd) AS observed_gateway_cost_usd,
             APPROX_QUANTILES(latency_ms, 100)[OFFSET(50)] AS p50_latency_ms
      FROM `{PROJECT_ID}.{DATASET}.so20_results`
      GROUP BY run_id, model
      ORDER BY run_id
    ''').to_dataframe()
    display(live_summary)
else:
    print("Published aggregate results are shown below / 公開済み集計結果を以下に表示")

Published aggregate results are shown below / 公開済み集計結果を以下に表示


In [6]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
summary_path = repo_root / "results" / "summary.csv"
performance_path = repo_root / "results" / "performance.csv"
raw_base = "https://raw.githubusercontent.com/jackojacko05/compare-jev-bigquery-ai-functions/main/results"

# Use checked-in files locally and the same public files when run as a Colab scratchpad.
results = pd.read_csv(summary_path if summary_path.exists() else f"{raw_base}/summary.csv")
performance = pd.read_csv(performance_path if performance_path.exists() else f"{raw_base}/performance.csv")
results[["method", "accuracy", "correct", "total", "theoretical_model_cost_usd"]].tail(10)

,method,accuracy,correct,total,theoretical_model_cost_usd
5,Jev one-shot per label,0.860,172.0,200.0,0.009674
6,Jev ambiguous rerank,0.850,170.0,200.0,0.011565
7,BQML Random Forest,NaN,NaN,NaN,NaN
8,Jev + Gemini 3.8 Flash rerank,0.865,173.0,200.0,0.066046
9,Gemini 2.5 Flash-Lite + Gemini 3.8 Flash rerank,0.855,171.0,200.0,0.075124
10,Gemini 3.8 Flash full,0.870,174.0,200.0,0.158212
11,Jev + Gemini 3.1 Pro rerank,0.860,172.0,200.0,0.275126
12,Gemini 2.5 Flash-Lite + Gemini 3.1 Pro rerank,0.850,170.0,200.0,0.278731
13,Gemini 3.1 Pro full,0.885,177.0,200.0,0.550554
14,Gemini 2.5 Flash-Lite via Gateway + Gemini 3.1...,0.845,169.0,200.0,0.276779


## 結果 / Results

経路を揃えたベースライン、最新Pro、2種類のPro再判定パイプラインを掲載します。

The table covers the route-equalized baselines, latest Pro, and two Pro-rerank pipelines.

In [7]:
primary_methods = [
    "Jev baseline",
    "Gemini 2.5 Flash-Lite via Vercel AI Gateway",
    "Jev + Gemini 3.1 Pro rerank",
    "Gemini 2.5 Flash-Lite via Gateway + Gemini 3.1 Pro rerank",
    "Gemini 3.1 Pro full",
]
display_names = {
    "Jev baseline": "Jev",
    "Gemini 2.5 Flash-Lite via Vercel AI Gateway": "Gemini 2.5 Flash-Lite",
    "Jev + Gemini 3.1 Pro rerank": "Jev → Gemini 3.1 Pro再判定 / rerank",
    "Gemini 2.5 Flash-Lite via Gateway + Gemini 3.1 Pro rerank": "Gemini 2.5 Flash-Lite → Gemini 3.1 Pro再判定 / rerank",
    "Gemini 3.1 Pro full": "Gemini 3.1 Pro",
}
primary = (
    results[results["method"].isin(primary_methods)]
    .assign(accuracy_pct=lambda x: x["accuracy"] * 100,
            cost_per_1000_usd=lambda x: x["theoretical_model_cost_usd"] * 5)
    [["method", "accuracy_pct", "correct", "total", "theoretical_model_cost_usd", "cost_per_1000_usd"]]
    .sort_values("theoretical_model_cost_usd")
)
primary_display = primary.assign(method=lambda x: x["method"].map(display_names)).rename(columns={
    "method": "手法 / Method", "accuracy_pct": "正解率 (%) / Accuracy (%)",
    "correct": "正解数 / Correct", "total": "件数 / Total",
    "theoretical_model_cost_usd": "理論費用 (USD) / Theoretical cost",
    "cost_per_1000_usd": "1,000件費用 (USD) / Cost per 1,000",
})
primary_display

/Users/user/.cache/uv/archive-v0/sOfdzlp-SWMHtH7E/lib/python3.14/site-packages/pandas/core/frame.py:5246: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data[k] = com.apply_if_callable(v, data)
/Users/user/.cache/uv/archive-v0/sOfdzlp-SWMHtH7

,手法 / Method,正解率 (%) / Accuracy (%),正解数 / Correct,件数 / Total,理論費用 (USD) / Theoretical cost,"1,000件費用 (USD) / Cost per 1,000"
0,Jev,85.5,171.0,200.0,0.007882,0.039408
2,Gemini 2.5 Flash-Lite,86.0,172.0,200.0,0.015539,0.077696
11,Jev → Gemini 3.1 Pro再判定 / rerank,86.0,172.0,200.0,0.275126,1.375628
14,Gemini 2.5 Flash-Lite → Gemini 3.1 Pro再判定 / re...,84.5,169.0,200.0,0.276779,1.383895
13,Gemini 3.1 Pro,88.5,177.0,200.0,0.550554,2.752770


In [8]:
import numpy as np
import plotly.graph_objects as go

tradeoff = (
    primary[primary["method"].isin([
        "Jev baseline", "Gemini 2.5 Flash-Lite via Vercel AI Gateway", "Gemini 3.1 Pro full",
    ])]
    .merge(performance[["method", "p50_latency_ms"]], on="method", how="inner")
    .assign(label=lambda x: x["method"].map(display_names))
)

def higher_is_better(series):
    span = series.max() - series.min()
    return (series - series.min()) / span if span else series * 0 + 0.5

def lower_is_better(series):
    return 1 - higher_is_better(series)

# 色は列ごとの相対的な良さ（濃いほど有利）、文字は比較可能な実値です。
# Color is relative desirability within each column; annotations retain raw values.
scores = np.column_stack([
    higher_is_better(tradeoff["accuracy_pct"]),
    lower_is_better(np.log10(tradeoff["cost_per_1000_usd"])),
    lower_is_better(np.log10(tradeoff["p50_latency_ms"])),
])
raw_text = np.column_stack([
    tradeoff["accuracy_pct"].map(lambda v: f"{v:.1f}%"),
    tradeoff["cost_per_1000_usd"].map(lambda v: f"${v:.3f}"),
    tradeoff["p50_latency_ms"].map(lambda v: f"{v:,.0f} ms"),
])

heatmap_fig = go.Figure(go.Heatmap(
    z=scores,
    x=["精度 / Accuracy", "費用（安いほど良い）/ Cost", "p50（速いほど良い）/ Latency"],
    y=tradeoff["label"],
    text=raw_text,
    texttemplate="%{text}",
    textfont={"size": 15},
    colorscale=[[0, "#EEF2FF"], [0.5, "#60A5FA"], [1, "#0F766E"]],
    zmin=0,
    zmax=1,
    colorbar={"title": "列内の相対評価<br>Relative score", "tickvals": [0, 1], "ticktext": ["低 / Low", "高 / High"]},
    hovertemplate="%{y}<br>%{x}<br>実値 / Raw: %{text}<br>相対評価 / Relative: %{z:.2f}<extra></extra>",
))
heatmap_fig.update_layout(
    title="精度・費用・速度のトレードオフ（200件） / Accuracy, cost, and speed trade-off",
    height=430,
    margin={"l": 190, "r": 100, "t": 90, "b": 70},
    xaxis={"side": "top"},
)
heatmap_fig.show()
heatmap_fig.write_html("blog-tradeoff-heatmap.html", include_plotlyjs="cdn")

/Users/user/.cache/uv/archive-v0/sOfdzlp-SWMHtH7E/lib/python3.14/site-packages/pandas/core/frame.py:5246: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data[k] = com.apply_if_callable(v, data)


**図の読み方 / How to read:** 各列で濃いほど有利です。色は列内で正規化した相対評価なので、差の大きさはセル内の実値で確認してください。再判定パイプラインは比較可能なp50がないため表だけに掲載します。 / Darker is better within each column. Colors are column-normalized, so use the raw annotation to judge magnitude. Rerank pipelines remain table-only because comparable p50 latency was not recorded.

![Static heatmap / 静的ヒートマップ](https://raw.githubusercontent.com/jackojacko05/compare-jev-bigquery-ai-functions/main/docs/assets/blog-tradeoff-heatmap.png)

### 解釈 / Interpretation

JevとGemini 2.5 Flash-Liteの差は200件中1問です。一方、Jevの費用は約半分、p50は約3.6倍高速でした。Gemini 3.1 Proが最も高精度ですが、費用はJevの約70倍です。広い再判定は第一段階の正解を誤答へ変える場合もありました。

Jev and Gemini 2.5 Flash-Lite differ by one correct answer in 200 rows, while Jev costs about half as much and is about 3.6× faster at p50. Gemini 3.1 Pro is most accurate but costs about 70× Jev. Broad reranking can replace correct first-stage answers with wrong ones.

## 検証 / Checks

In [9]:
assert set(primary_methods) == set(primary["method"])
assert len(primary) == 5
assert len(tradeoff) == 3
assert (primary["total"] == 200).all()
assert primary["accuracy_pct"].between(0, 100).all()
assert primary["theoretical_model_cost_usd"].gt(0).all()

jev = primary.loc[primary["method"] == "Jev baseline"].iloc[0]
flash_lite = primary.loc[primary["method"] == "Gemini 2.5 Flash-Lite via Vercel AI Gateway"].iloc[0]
pro = primary.loc[primary["method"] == "Gemini 3.1 Pro full"].iloc[0]

checks = {
    "Jev正解数 / Jev correct": int(jev["correct"]),
    "Flash-Lite正解数 / correct": int(flash_lite["correct"]),
    "Pro正解数 / Pro correct": int(pro["correct"]),
    "Jev費用削減率 / cost reduction vs Flash-Lite": 1 - jev["theoretical_model_cost_usd"] / flash_lite["theoretical_model_cost_usd"],
    "ProとJevの費用倍率 / Pro-to-Jev cost multiple": pro["theoretical_model_cost_usd"] / jev["theoretical_model_cost_usd"],
}
checks

{'Jev正解数 / Jev correct': 171,
 'Flash-Lite正解数 / correct': 172,
 'Pro正解数 / Pro correct': 177,
 'Jev費用削減率 / cost reduction vs Flash-Lite': np.float64(0.4927865835215681),
 'ProとJevの費用倍率 / Pro-to-Jev cost multiple': np.float64(69.85272100730305)}

## まとめ / Takeaways

この20ラベル分類では、JevはGemini 2.5 Flash-Liteにほぼ並ぶ精度を、より低い費用で達成しました。Proの改善は数問に留まり、今回の広い二段階判定は選択性が不足しています。数十万件へ広げる前に、再試行や行ごとのプロンプト反復を含む経路全体を測ることが重要です。

For this closed 20-label task, Jev nearly matches Gemini 2.5 Flash-Lite at materially lower cost. Pro improves only a few questions, and the broad two-stage rule is not selective enough. Test the full path—including retries and per-row prompt repetition—before scaling.

これは200件のパイロットです。タグは観測されたユーザーラベルであり、客観的な正解ではありません。価格も変わり得ます。 / This is a 200-row pilot. Tags are observed user labels, not objective ground truth, and prices can change.